# Hands-on 5: Redes Neuronales Convolucionales (CNN) con Python y TensorFlow/Keras (Dataset MNIST)

**Asignatura:** Deep Learning  
**Integrantes:** Iker Gael Rico Nonato 
  Luis Eduardo Navarro Hernandez 

En este tutorial implementaremos **paso a paso** una **Red Neuronal Convolucional (CNN)** utilizando TensorFlow y Keras en Google Colab / Jupyter. Analizaremos tanto los fundamentos teóricos y matemáticos como cada una de las etapas que conforman una CNN: **Convolución**, **ReLU**, **Max Pooling**, **Flattening**, **Dense**, **Evaluation** y **Prediction**.


---
## 1. Fundamentos Teóricos y Matemáticos de las CNN

### 1.1 ¿Por qué usar una CNN en lugar de un MLP para imágenes?
Un **Perceptrón Multicapa (MLP)** tradicional trata una imagen como un vector plano unidimensional: por ejemplo, una imagen de $28 \times 28$ píxeles se aplana desde el inicio a un vector de $784$ valores ($X \in \mathbb{R}^{28 \times 28} \rightarrow x \in \mathbb{R}^{784}$). Esto provoca dos problemas graves:
1. **Pérdida de estructura espacial:** Al aplanar al inicio, el modelo pierde la noción de vecindad entre píxeles (bordes, líneas, curvas o formas).
2. **Explosión de parámetros:** Si una imagen de $100 \times 100 \times 3$ entra a una capa densa de $100$ neuronas, se requieren $3{,}000{,}000$ de pesos.

En contraste, una **CNN** trabaja directamente con tensores tridimensionales ($X \in \mathbb{R}^{28 \times 28 \times 1}$) conservando la posición vertical, horizontal y el canal de intensidad. Utiliza **filtros (kernels) pequeños con pesos compartidos** y conectividad local, reduciendo drásticamente el número de parámetros y logrando invarianza espacial parcial.

### 1.2 Arquitectura General y Formulación Matemática

El flujo clásico de una CNN sigue la secuencia:
$$\text{Imagen} \rightarrow \text{Convolución} \rightarrow \text{ReLU} \rightarrow \text{Max Pooling} \rightarrow \dots \rightarrow \text{Flattening} \rightarrow \text{Dense} \rightarrow \text{Softmax}$$

1. **Convolución 2D:** Aplica un filtro $K \in \mathbb{R}^{k \times k}$ sobre la imagen o mapa de características $I \in \mathbb{R}^{m \times n}$ para extraer patrones locales:
   $$(I * K)(i, j) = \sum_{u=0}^{k-1} \sum_{v=0}^{k-1} I(i + u, j + v) \cdot K(u, v) + b$$

2. **Activación ReLU (*Rectified Linear Unit*):** Se aplica elemento a elemento para introducir no linealidad, anulando valores negativos y conservando las activaciones positivas significativas:
   $$\text{ReLU}(x) = \max(0, x)$$

3. **Max Pooling:** Reduce la dimensionalidad espacial (ancho y alto) conservando el valor máximo dentro de cada subregión o ventana $R \subset \mathbb{R}^{h \times w}$:
   $$P = \max_{(i,j) \in R} x_{i,j}$$

4. **Flattening (Aplanamiento):** Transforma el tensor tridimensional de mapas de características ($H \times W \times C$) en un vector unidimensional $x \in \mathbb{R}^{H \cdot W \cdot C}$ para conectarlo con el clasificador denso.

5. **Capas Densas (*Fully Connected*) y Softmax:** Integran globalmente las características extraídas mediante combinaciones lineales $z = Wx + b$ y convierten los *logits* finales en una distribución de probabilidad sobre las $C=10$ clases:
   $$\hat{y}_i = \frac{e^{z_i}}{\sum_{j=1}^{C} e^{z_j}}$$

6. **Función de Pérdida (*Categorical Cross-Entropy*):** Mide la discrepancia entre la distribución real $y$ (en formato *one-hot*) y la predicha $\hat{y}$:
   $$L = -\sum_{i=1}^{C} y_i \cdot \log(\hat{y}_i)$$


---
## 2. Preparación del Entorno y Patrones de Entrenamiento (Dataset MNIST)

### 2.1 Importación de Librerías
Cargamos TensorFlow/Keras para construir la CNN, NumPy para manejo de arreglos y Matplotlib/Seaborn para visualizar imágenes y métricas.


In [ ]:
# Importación de NumPy para operaciones numéricas y manejo de arreglos multidimensionales
import numpy as np

# Importación de Matplotlib y Seaborn para visualización de imágenes, curvas y matriz de confusión
import matplotlib.pyplot as plt
import seaborn as sns

# Importación de TensorFlow y módulos de Keras para construir la Red Neuronal Convolucional
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical

# Importación de métricas detalladas de Scikit-Learn para la etapa de evaluación
from sklearn.metrics import confusion_matrix, classification_report

# Confirmación de la versión de TensorFlow cargada
print(f"TensorFlow versión: {tf.__version__} - Librerías importadas correctamente.")


### 2.2 Carga, Exploración y Preprocesamiento del Dataset MNIST
El dataset **MNIST** contiene $60{,}000$ imágenes de entrenamiento y $10{,}000$ de prueba, cada una de $28 \times 28$ píxeles en escala de grises (valores de $0$ a $255$) correspondientes a dígitos del `0` al `9`.

Para alimentar correctamente a la CNN realizamos tres pasos clave:
1. **Normalización:** Dividimos los píxeles entre `255.0` para escalarlos al rango $[0, 1]$ en formato `float32`.
2. **Redimensionamiento (`reshape`):** Agregamos explícitamente la dimensión del canal (`1` por ser escala de grises), pasando de `(60000, 28, 28)` a `(60000, 28, 28, 1)` que es el formato `(batch, altura, ancho, canales)` esperado por `Conv2D`.
3. **Codificación One-Hot (`to_categorical`):** Convertimos las etiquetas enteras (ej. `4`) en vectores binarios de 10 posiciones para compararlas con la salida `softmax` usando `categorical_crossentropy`.


In [ ]:
# Carga automática del dataset MNIST dividido en conjuntos de entrenamiento y prueba
(x_train_raw, y_train_raw), (x_test_raw, y_test_raw) = mnist.load_data()

# Impresión de las dimensiones originales de los conjuntos cargados
print(f"Forma original x_train: {x_train_raw.shape}, y_train: {y_train_raw.shape}")
print(f"Forma original x_test:  {x_test_raw.shape}, y_test:  {y_test_raw.shape}")

# Visualización de los primeros 10 dígitos del conjunto de entrenamiento
plt.figure(figsize=(10, 4))
for i in range(10):
    # Crea una subgráfica de 2 filas por 5 columnas
    plt.subplot(2, 5, i + 1)
    # Muestra la imagen en escala de grises
    plt.imshow(x_train_raw[i], cmap="gray")
    # Coloca como título la etiqueta real del dígito
    plt.title(f"Dígito: {y_train_raw[i]}")
    # Oculta los ejes coordenados para mayor claridad visual
    plt.axis("off")
plt.suptitle("Muestras del Dataset MNIST", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# 1. Normalización: convertir a float32 y escalar valores de [0, 255] a [0.0, 1.0]
x_train = x_train_raw.astype("float32") / 255.0
x_test = x_test_raw.astype("float32") / 255.0

# 2. Redimensionamiento: agregar el canal de color (1 = escala de grises) para Conv2D
x_train = x_train.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)

# 3. Codificación One-Hot: transformar etiquetas enteras (0-9) en vectores binarios de longitud 10
y_train = to_categorical(y_train_raw, num_classes=10)
y_test = to_categorical(y_test_raw, num_classes=10)

# Verificación de las nuevas dimensiones listas para la CNN
print(f"\nForma procesada x_train: {x_train.shape} | y_train (One-Hot): {y_train.shape}")
print(f"Forma procesada x_test:  {x_test.shape} | y_test (One-Hot):  {y_test.shape}")


---
## 3. Construcción de la CNN: Etapa por Etapa

Construiremos el modelo secuencial (`Sequential`) paso a paso, explicando y agregando cada capa requerida en su propia sección.


### 3.1 Etapa 1: Convolución (`Conv2D`)
La capa convolucional aplica filtros (kernels) entrenables sobre la imagen de entrada para detectar características locales como bordes horizontales, verticales, diagonales y curvas.
* Definimos la entrada explícita con `Input(shape=(28, 28, 1))`.
* Agregamos la primera capa `Conv2D` con **32 filtros** de tamaño **$3 \times 3$** (sin función de activación aún, para mostrar explícitamente la etapa de **ReLU** en la siguiente sección).
* Con un kernel de $3 \times 3$ sin *padding* y *stride* 1, una entrada de $28 \times 28$ produce mapas de características de $26 \times 26 \times 32$.


In [ ]:
# Inicializamos un modelo secuencial vacío donde iremos apilando cada etapa de la CNN
model = Sequential(name="CNN_MNIST")

# Definimos la capa de entrada especificando las dimensiones de cada imagen (28x28x1)
model.add(Input(shape=(28, 28, 1)))

# Agregamos la primera capa de Convolución 2D: 32 filtros de tamaño 3x3
# Se omite 'activation' aquí para aplicar ReLU explícitamente como la siguiente etapa
model.add(Conv2D(filters=32, kernel_size=(3, 3), name="conv_1"))

# Mostramos la forma de salida hasta esta etapa: (None, 26, 26, 32)
print(f"Salida tras Convolución 1: {model.output_shape}")


### 3.2 Etapa 2: Función de Activación ReLU
La función **ReLU** ($\text{ReLU}(x) = \max(0, x)$) se aplica elemento a elemento sobre el mapa de características generado por la convolución. Su propósito es:
* Introducir **no linealidad** en la red.
* Anular valores negativos (silenciar regiones irrelevantes) y resaltar únicamente las activaciones positivas significativas.
* Acelerar la convergencia y mitigar el problema del gradiente desvanecido.


In [ ]:
# Importamos la capa Activation de Keras para agregar ReLU de forma explícita
from tensorflow.keras.layers import Activation

# Agregamos la capa de activación no lineal ReLU sobre los mapas convolucionados
model.add(Activation("relu", name="relu_1"))

# La dimensión espacial se mantiene igual tras ReLU: (None, 26, 26, 32)
print(f"Salida tras ReLU 1: {model.output_shape}")


### 3.3 Etapa 3: Max Pooling (`MaxPooling2D`)
La capa de **Max Pooling** reduce la resolución espacial (altura y anchura) de los mapas de activación tomando el valor máximo en ventanas de **$2 \times 2$** con desplazamiento (*stride*) de 2. Esto logra:
* Reducir el número de parámetros y el costo computacional.
* Prevenir el sobreajuste (*overfitting*).
* Aportar invarianza espacial parcial ante pequeñas traslaciones del dígito.

Para extraer patrones más complejos y abstractos, también añadiremos un **segundo bloque** (`Conv2D` de 64 filtros con `relu` + `MaxPooling2D`).


In [ ]:
# Agregamos la primera capa de Max Pooling con ventana de 2x2 (reduce 26x26 a 13x13)
model.add(MaxPooling2D(pool_size=(2, 2), name="maxpool_1"))
print(f"Salida tras Max Pooling 1: {model.output_shape}")

# Segundo bloque convolucional: 64 filtros de 3x3 con activación ReLU integrada
# Al aumentar la profundidad, la red detecta formas y trazos más complejos
model.add(Conv2D(filters=64, kernel_size=(3, 3), activation="relu", name="conv_2_relu"))
print(f"Salida tras Convolución 2 + ReLU: {model.output_shape}")

# Segunda capa de Max Pooling 2x2 (reduce 11x11 a 5x5)
model.add(MaxPooling2D(pool_size=(2, 2), name="maxpool_2"))
print(f"Salida tras Max Pooling 2: {model.output_shape}")


### 3.4 Etapa 4: Flattening (`Flatten`)
Tras las etapas de extracción de características espaciales, obtenemos un tensor 3D de dimensiones $(5 \times 5 \times 64)$. La capa **`Flatten()`** convierte este volumen tridimensional en un **vector plano unidimensional** de $5 \times 5 \times 64 = 1{,}600$ elementos para poder conectarlo con las capas totalmente conectadas (*Dense*).


In [ ]:
# Agregamos la capa Flatten para transformar el tensor 3D (5, 5, 64) en un vector 1D de 1600 elementos
model.add(Flatten(name="flatten"))

# Verificamos la nueva forma unidimensional de salida: (None, 1600)
print(f"Salida tras Flattening: {model.output_shape}")


### 3.5 Etapa 5: Capas Densas (`Dense`), Compilación y Entrenamiento
En esta etapa integramos globalmente las características extraídas para clasificar el dígito:
* **`Dense(64, activation='relu')`:** Capa oculta totalmente conectada con 64 neuronas y activación ReLU que aprende combinaciones no lineales de las características aplanadas.
* **`Dense(10, activation='softmax')`:** Capa de salida con 10 neuronas (una por cada dígito del 0 al 9). La función **Softmax** transforma las salidas en probabilidades que suman $1$.

Posteriormente compilamos el modelo con el optimizador **`adam`**, la función de pérdida **`categorical_crossentropy`** y la métrica **`accuracy`**, y lo entrenamos sobre `x_train` y `y_train`.


In [ ]:
# Agregamos una capa oculta totalmente conectada (Dense) de 64 neuronas con activación ReLU
model.add(Dense(units=64, activation="relu", name="dense_oculta"))

# Agregamos Dropout (25%) opcional para regularizar y evitar sobreajuste durante el entrenamiento
model.add(Dropout(rate=0.25, name="dropout"))

# Agregamos la capa de salida Dense con 10 neuronas (dígitos 0-9) y activación Softmax
model.add(Dense(units=10, activation="softmax", name="dense_salida_softmax"))

# Imprimimos el resumen completo de la arquitectura de la CNN y sus parámetros entrenables
model.summary()


In [ ]:
# Compilación del modelo definiendo el optimizador, la función de pérdida y la métrica
model.compile(
    optimizer="adam",                      # Optimizador adaptativo Adam para ajustar los pesos
    loss="categorical_crossentropy",       # Pérdida para clasificación multiclase con etiquetas One-Hot
    metrics=["accuracy"]                   # Métrica de exactitud porcentual
)

# Entrenamiento de la CNN durante 5 épocas con lotes de 64 imágenes y 10% para validación interna
print("Iniciando entrenamiento de la CNN...")
history = model.fit(
    x_train,                               # Imágenes de entrenamiento normalizadas
    y_train,                               # Etiquetas reales en formato One-Hot
    epochs=5,                              # Número de pasadas completas sobre el dataset
    batch_size=64,                         # Cantidad de imágenes procesadas antes de actualizar pesos
    validation_split=0.1,                  # Reserva 10% de train para monitorear generalización por época
    verbose=1                              # Muestra la barra de progreso y métricas por época
)
print("Entrenamiento finalizado.")


### 3.6 Etapa 6: Evaluation (Evaluación del Modelo)
Evaluamos el desempeño real de la CNN sobre el conjunto de prueba (`x_test`, `y_test`), el cual contiene $10{,}000$ imágenes que el modelo nunca vio durante el entrenamiento. Además, graficamos las **curvas de aprendizaje** (*Accuracy* y *Loss*), la **Matriz de Confusión** y el **Reporte de Clasificación** por dígito.


In [ ]:
# Evaluamos el modelo sobre el conjunto de prueba completo para obtener pérdida y exactitud finales
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f"==================================================")
print(f" PÉRDIDA EN TEST (Loss):       {test_loss:.4f}")
print(f" EXACTITUD EN TEST (Accuracy): {test_accuracy * 100:.2f}%")
print(f"==================================================\n")

# Graficamos la evolución de Accuracy y Loss durante entrenamiento y validación
plt.figure(figsize=(12, 4))

# Subgráfica 1: Exactitud (Accuracy) por época
plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Entrenamiento", marker="o")
plt.plot(history.history["val_accuracy"], label="Validación", marker="s")
plt.title("Curva de Exactitud (Accuracy)", fontweight="bold")
plt.xlabel("Época")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)

# Subgráfica 2: Pérdida (Loss) por época
plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Entrenamiento", marker="o")
plt.plot(history.history["val_loss"], label="Validación", marker="s")
plt.title("Curva de Pérdida (Loss)", fontweight="bold")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

# Generamos las predicciones de probabilidad sobre todo el conjunto de prueba
y_pred_probs = model.predict(x_test, verbose=0)
# Obtenemos la clase predicha (índice de mayor probabilidad) y la clase real
y_pred_classes = np.argmax(y_pred_probs, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

# Calculamos e imprimimos el reporte de clasificación (Precision, Recall, F1-Score por dígito)
print("--- REPORTE DE CLASIFICACIÓN POR DÍGITO ---")
print(classification_report(y_true_classes, y_pred_classes))

# Calculamos y graficamos la Matriz de Confusión
cm = confusion_matrix(y_true_classes, y_pred_classes)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=range(10), yticklabels=range(10))
plt.title("Matriz de Confusión - CNN sobre MNIST", fontsize=13, fontweight="bold")
plt.xlabel("Dígito Predicho")
plt.ylabel("Dígito Real")
plt.show()


### 3.7 Etapa 7: Prediction (Predicción de Muestras)
Finalmente, utilizamos el modelo entrenado para realizar inferencia:
1. **Predicción individual:** Seleccionamos una imagen específica del conjunto de prueba (`index = 40`), adaptamos su forma a `(1, 28, 28, 1)`, calculamos el vector de 10 probabilidades con `model.predict()` y extraemos la clase más probable con `np.argmax()`.
2. **Visualización de múltiples predicciones:** Mostramos varias imágenes del conjunto de prueba comparando la predicción de la CNN contra su etiqueta real.


In [ ]:
# --- 1. PREDICCIÓN INDIVIDUAL ---
# Seleccionamos el índice de un patrón de prueba específico
index = 40

# Adaptamos la imagen a las dimensiones esperadas por la CNN: (1 muestra, 28 alto, 28 ancho, 1 canal)
sample = x_test[index].reshape(1, 28, 28, 1)

# Realizamos la predicción obteniendo un vector con las 10 probabilidades (una por dígito)
prediction_probs = model.predict(sample, verbose=0)

# Extraemos el índice con la probabilidad más alta (etiqueta predicha) y su confianza
predicted_label = np.argmax(prediction_probs)
confidence = np.max(prediction_probs) * 100
true_label = y_test_raw[index]

# Mostramos el vector de probabilidades redondeado
print(f"Probabilidades predichas (0-9): {np.round(prediction_probs[0], 3)}")
print(f"Predicción: {predicted_label} (Confianza: {confidence:.2f}%) | Etiqueta Real: {true_label}")

# Visualizamos la imagen evaluada
plt.figure(figsize=(3.5, 3.5))
plt.imshow(x_test[index].reshape(28, 28), cmap="gray")
plt.title(f"Predicción: {predicted_label} | Real: {true_label}", fontweight="bold")
plt.axis("off")
plt.show()

# --- 2. VISUALIZACIÓN DE MÚLTIPLES PREDICCIONES ---
# Configuramos una figura con 10 imágenes de prueba (2 filas x 5 columnas)
plt.figure(figsize=(11, 4.5))
for i in range(10):
    # Extraemos la imagen i-ésima y su etiqueta real
    image = x_test[i]
    real = y_test_raw[i]
    # Predecimos el dígito usando argmax sobre la salida de model.predict
    pred = np.argmax(model.predict(image.reshape(1, 28, 28, 1), verbose=0))
    
    # Dibujamos la subgráfica correspondiente
    plt.subplot(2, 5, i + 1)
    plt.imshow(image.reshape(28, 28), cmap="gray")
    # Coloreamos el título en verde si acertó, o rojo si falló
    color = "green" if pred == real else "red"
    plt.title(f"Pred: {pred} | Real: {real}", color=color, fontweight="bold")
    plt.axis("off")

plt.suptitle("Predicciones de la CNN sobre Muestras de Prueba MNIST", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


---
## 4. Conclusiones
1. **Extracción Jerárquica de Características:** Las capas `Conv2D` y `ReLU` permiten identificar trazos locales (bordes y curvas) que se combinan en patrones más complejos en capas más profundas, superando las limitaciones de un MLP que aplana la imagen desde el inicio.
2. **Eficiencia Espacial:** `MaxPooling2D` reduce las dimensiones espaciales conservando las activaciones más fuertes, mientras que `Flatten` sirve de puente hacia el clasificador `Dense`.
3. **Desempeño en MNIST:** Con pocas épocas de entrenamiento, esta arquitectura CNN alcanza típicamente alrededor de **99% de exactitud** sobre el conjunto de prueba MNIST.
